# 🗄️ Database Explorer — Not Yet Priced In
Quick notebook to verify and query the SQLite database.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("../data/not_yet_priced_in.db")
conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH} ({DB_PATH.stat().st_size / 1024 / 1024:.2f} MB)")

Connected to: ../data/not_yet_priced_in.db (2.12 MB)


## 1. Table Overview

In [2]:
# List all tables and row counts
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for t in tables['name']:
    count = pd.read_sql(f"SELECT COUNT(*) as rows FROM {t}", conn).iloc[0, 0]
    print(f"  {t:20s} → {count:,} rows")

  filings              → 942 rows
  sqlite_sequence      → 1 rows
  features             → 942 rows
  reactions            → 942 rows


## 2. Schema Inspection

In [3]:
# Show columns for each table
for t in ['filings', 'features', 'reactions']:
    cols = pd.read_sql(f"PRAGMA table_info({t})", conn)
    print(f"\n📋 {t}:")
    print(cols[['name', 'type', 'notnull']].to_string(index=False))


📋 filings:
            name      type  notnull
              id   INTEGER        0
       accession      TEXT        1
          ticker      TEXT        1
         company      TEXT        0
      filed_date      TEXT        0
            year   INTEGER        0
    primary_item      TEXT        0
  broad_category      TEXT        0
      clean_text      TEXT        0
parse_confidence      REAL        0
      created_at TIMESTAMP        0

📋 features:
                    name    type  notnull
               filing_id INTEGER        0
         numeric_density    REAL        0
 forward_looking_density    REAL        0
financial_symbol_density    REAL        0
     baseline_importance    REAL        0
        importance_score INTEGER        0
      llm_event_category    TEXT        0
          llm_confidence    REAL        0
          grounding_rate    REAL        0
             key_signals    TEXT        0
               reasoning    TEXT        0

📋 reactions:
             name    type

## 3. Data by Year

In [4]:
pd.read_sql("""
    SELECT year, COUNT(*) as filings,
           COUNT(DISTINCT ticker) as tickers
    FROM filings
    GROUP BY year
    ORDER BY year
""", conn)

,year,filings,tickers
0,2021,25,3
1,2022,443,36
2,2023,474,42


## 4. Reaction Class Distribution

In [5]:
df_reactions = pd.read_sql("""
    SELECT r.reaction_class, 
           COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM reactions), 1) as pct
    FROM reactions r
    GROUP BY r.reaction_class
    ORDER BY count DESC
""", conn)
df_reactions

,reaction_class,count,pct
0,Immediate,539,57.2
1,No Reaction,173,18.4
2,Gradual,128,13.6
3,Delayed,74,7.9
4,Unknown,28,3.0


## 5. Reaction Class by Year (Cross-Tab)

In [6]:
pd.read_sql("""
    SELECT f.year, r.reaction_class, COUNT(*) as count
    FROM filings f
    JOIN reactions r ON f.id = r.filing_id
    GROUP BY f.year, r.reaction_class
    ORDER BY f.year, count DESC
""", conn).pivot_table(index='reaction_class', columns='year', values='count', fill_value=0)

year,2021,2022,2023
reaction_class,,,
Delayed,1.0,38.0,35.0
Gradual,1.0,28.0,99.0
Immediate,18.0,283.0,238.0
No Reaction,5.0,67.0,101.0
Unknown,0.0,27.0,1.0


## 6. Top Tickers by Delayed Reactions

In [7]:
pd.read_sql("""
    SELECT f.ticker, 
           COUNT(*) as total_filings,
           SUM(CASE WHEN r.reaction_class = 'Delayed' THEN 1 ELSE 0 END) as delayed,
           ROUND(SUM(CASE WHEN r.reaction_class = 'Delayed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as delayed_pct
    FROM filings f
    JOIN reactions r ON f.id = r.filing_id
    GROUP BY f.ticker
    HAVING delayed > 0
    ORDER BY delayed DESC
    LIMIT 15
""", conn)

,ticker,total_filings,delayed,delayed_pct
0,SBUX,28,6,21.4
1,T,37,6,16.2
2,SLB,27,5,18.5
3,AXP,50,4,8.0
4,MMM,41,4,9.8
5,UNP,25,4,16.0
6,ABBV,31,3,9.7
7,CVX,25,3,12.0
8,HON,26,3,11.5
9,TSLA,24,3,12.5


## 7. Feature Statistics

In [8]:
pd.read_sql("""
    SELECT 
        ROUND(AVG(numeric_density), 2) as avg_numeric_density,
        ROUND(AVG(forward_looking_density), 2) as avg_fld,
        ROUND(AVG(baseline_importance), 3) as avg_baseline_imp,
        ROUND(AVG(importance_score), 2) as avg_llm_score,
        ROUND(AVG(grounding_rate), 3) as avg_grounding
    FROM features
""", conn)

,avg_numeric_density,avg_fld,avg_baseline_imp,avg_llm_score,avg_grounding
0,9.09,1.09,0.4,3.52,0.831


## 8. Features vs Reaction Class (The Signal)

In [9]:
pd.read_sql("""
    SELECT r.reaction_class,
           COUNT(*) as n,
           ROUND(AVG(ft.numeric_density), 2) as avg_nd,
           ROUND(AVG(ft.forward_looking_density), 2) as avg_fld,
           ROUND(AVG(ft.importance_score), 2) as avg_llm_score,
           ROUND(AVG(ft.baseline_importance), 3) as avg_baseline,
           ROUND(AVG(r.MOS_prospective), 3) as avg_mos
    FROM reactions r
    JOIN features ft ON r.filing_id = ft.filing_id
    WHERE r.reaction_class IS NOT NULL AND r.reaction_class != 'Unknown'
    GROUP BY r.reaction_class
    ORDER BY avg_mos DESC
""", conn)

,reaction_class,n,avg_nd,avg_fld,avg_llm_score,avg_baseline,avg_mos
0,Delayed,74,9.21,1.27,3.44,0.413,0.679
1,No Reaction,173,9.35,1.39,3.52,0.444,0.546
2,Gradual,128,8.18,1.14,3.48,0.432,0.496
3,Immediate,539,9.35,0.94,3.55,0.382,0.440


## 9. ML Training Data Preview

In [10]:
# This is the query you'll use to build your ML feature matrix
df_ml = pd.read_sql("""
    SELECT 
        f.year, f.ticker, f.broad_category,
        ft.numeric_density, ft.forward_looking_density, 
        ft.financial_symbol_density, ft.baseline_importance,
        ft.importance_score, ft.llm_confidence, ft.grounding_rate,
        r.reaction_class, r.R_short_0_1, r.R_long_5_20, r.MOS_prospective
    FROM filings f
    JOIN features ft ON f.id = ft.filing_id
    JOIN reactions r ON f.id = r.filing_id
    WHERE r.reaction_class IS NOT NULL AND r.reaction_class != 'Unknown'
""", conn)

print(f"ML-ready rows: {len(df_ml)}")
print(f"Train/Val (2021-2022): {len(df_ml[df_ml['year'].isin([2021, 2022])])}")
print(f"Test (2023): {len(df_ml[df_ml['year'] == 2023])}")
df_ml.head(10)

ML-ready rows: 914
Train/Val (2021-2022): 441
Test (2023): 473


,year,ticker,broad_category,numeric_density,forward_looking_density,financial_symbol_density,baseline_importance,importance_score,llm_confidence,grounding_rate,reaction_class,R_short_0_1,R_long_5_20,MOS_prospective
0,2023,AAPL,Earnings,8.490566,0.000000,0.000000,0.326653,4.0,NaN,1.000,Immediate,0.02297,0.03029,0.030
1,2023,AAPL,Earnings,8.490566,0.943396,0.000000,0.463080,5.0,NaN,0.667,Immediate,0.02760,0.00038,0.030
2,2023,AAPL,Earnings,8.490566,0.000000,0.000000,0.326653,4.0,NaN,0.000,Immediate,-0.06900,0.05443,0.030
3,2023,AAPL,Earnings,8.490566,0.000000,0.000000,0.326653,4.0,NaN,1.000,Gradual,-0.00172,-0.01058,0.030
4,2023,AAPL,Voting Results,21.103118,0.239808,0.000000,0.564698,3.0,NaN,1.000,No Reaction,0.01239,-0.01072,0.381
5,2023,AAPL,Regulatory,14.845361,3.917526,2.061856,0.902954,4.0,NaN,0.750,No Reaction,-0.00104,0.01391,0.844
6,2023,ABBV,Regulatory,8.219178,0.000000,0.000000,0.321378,1.0,NaN,0.000,Immediate,-0.05403,-0.04170,0.442
7,2023,ABBV,Material Agreement,6.481481,0.000000,0.925926,0.439522,4.0,NaN,0.667,Immediate,-0.02204,-0.06659,0.182
8,2023,ABBV,Voting Results,16.109422,0.607903,0.000000,0.574543,3.0,NaN,1.000,No Reaction,0.00826,-0.06878,0.381
9,2023,ABBV,Executive Change,5.882353,1.960784,0.000000,0.473980,5.0,NaN,1.000,No Reaction,-0.00010,0.06969,0.526


## 10. Sample Filing Deep Dive

In [11]:
# Pick a specific filing to inspect
sample = pd.read_sql("""
    SELECT f.accession, f.ticker, f.filed_date, f.broad_category,
           ft.importance_score, ft.key_signals, ft.reasoning,
           r.reaction_class, r.R_short_0_1, r.R_long_5_20
    FROM filings f
    JOIN features ft ON f.id = ft.filing_id
    JOIN reactions r ON f.id = r.filing_id
    WHERE r.reaction_class = 'Delayed'
    LIMIT 3
""", conn)

for _, row in sample.iterrows():
    print(f"\n{'='*60}")
    print(f"📄 {row['ticker']} — {row['filed_date']} ({row['broad_category']})")
    print(f"   LLM Score: {row['importance_score']}")
    print(f"   Reaction: {row['reaction_class']} (short={row['R_short_0_1']:.4f}, long={row['R_long_5_20']:.4f})")
    print(f"   Reasoning: {row['reasoning'][:200]}...")


📄 ABBV — 2023-10-12 (Executive Change)
   LLM Score: 3
   Reaction: Delayed (short=-0.0128, long=-0.0793)
   Reasoning: The filing announces changes in the company's Board of Directors with the appointment of two new directors and their respective committee assignments. These appointments could influence the company's ...

📄 ABBV — 2023-04-05 (Earnings)
   LLM Score: 4
   Reaction: Delayed (short=-0.0016, long=-0.0884)
   Reasoning: The filing discusses preliminary earnings results and provides specific guidance for both the first quarter and full-year 2023, with detailed earnings per share ranges. This information is significant...

📄 ABBV — 2023-10-27 (Earnings)
   LLM Score: 4
   Reaction: Delayed (short=-0.0022, long=-0.0679)
   Reasoning: The filing announces financial results for the third quarter, indicating it contains financials or guidance changes which are important updates. However, detailed figures or impacts are not cited, so ...


In [12]:
conn.close()
print("Connection closed.")

Connection closed.
